In [ ]:
# The cell 2 is to develop a deep learning model for the recommend system
# We can ignore it now

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#with open("/content/drive/MyDrive/3018 project/ALS version/als_model_package.pkl", "rb") as f:
    #data = pickle.load(f)

In [ ]:
!nvidia-smi

Sun Dec  7 10:29:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
# This is YOUR perfect parquet from Spark
parquet_path = "/content/drive/MyDrive/3018 project/meta_electronics.parquet"
df = pd.read_parquet(parquet_path)

print(f"Loaded {len(df):,} products from your Spark ETL")
print("Sample:")
display(df[["title", "brand", "price", "main_category"]].head(3))

Loaded 1,610,012 products from your Spark ETL
Sample:


,title,brand,price,main_category
0,FS-1051 FATSHARK TELEPORTER V3 HEADSET,Unknown,-1.00,All Electronics
1,Ce-H22B12-S1 4Kx2K Hdmi 4Port,Unknown,-1.00,All Electronics
2,Digi-Tatoo Decal Skin Compatible With MacBook ...,Digi-Tatoo,19.99,Computers


In [ ]:
# CELL 2 — FINAL DEEP LEARNING INDEX (run ONLY ONCE — takes ~2-3 min with GPU)
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
import os
import gc
from tqdm.notebook import tqdm

print("Loading your Spark parquet...")
df = pd.read_parquet("/content/drive/MyDrive/3018 project/meta_electronics.parquet")

# PERFECT PRICE HANDLING (-1 → safe)
df["price"] = pd.to_numeric(df["price"], errors='coerce')
df["has_price"] = df["price"].notna()
df["price"] = df["price"].fillna(-1)

# Clean text
df["title"] = df["title"].astype(str).str.lower().str.strip()
df["brand"] = df["brand"].astype(str).str.lower().str.strip()

# Keep only real products
df = df[df["has_price"] | (df["rating_number"] > 1000)].copy()

# Rank by quality (price is king)
df["quality_score"] = df["has_price"].astype(int)*100 + np.log1p(df["rating_number"])
df = df.sort_values("quality_score", ascending=False).head(1000000).reset_index(drop=True)

# Best possible search text
df["search_text"] = (
    df["title"] + " " +
    df["brand"] + " " +
    df["main_category"].fillna("") + " " +
    df["features_text"].fillna("") + " " +
    df["title"]
)

# Popularity
df["popularity"] = df["average_rating"] * np.log1p(df["rating_number"])

# DEEP LEARNING EMBEDDINGS (GPU fast!)
print("Loading Sentence Transformer with GPU...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

print("Encoding 1M products (2-3 min)...")
embeddings = model.encode(
    df["search_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Final product table
products = df[[
    "parent_asin", "title", "brand", "price", "average_rating",
    "rating_number", "main_category", "popularity", "has_price"
]].copy().reset_index(drop=True)

# SAVE FOREVER
save_path = "/content/drive/MyDrive/3018 project/recommend_index_DL_FINAL.pkl"

with open(save_path, "wb") as f:
    pickle.dump({
        "model": model,
        "embeddings": embeddings,
        "products": products
    }, f)

print(f"\nSUCCESS! Deep Learning index saved to:\n{save_path}")

Loading your Spark parquet...
Loading Sentence Transformer with GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Encoding 1M products (2-3 min)...


Batches:   0%|          | 0/3907 [00:00<?, ?it/s]


SUCCESS! Deep Learning index saved to:
/content/drive/MyDrive/3018 project/recommend_index_DL_FINAL.pkl


In [ ]:
# CELL 3 — FINAL INTERACTIVE DEEP LEARNING BOT (CLI + Telegram ready!)
import pickle
import re
from sklearn.metrics.pairwise import cosine_similarity
import threading

# Load your Deep Learning index (takes ~3 seconds)
print("Loading Deep Learning recommendation engine...")
with open("/content/drive/MyDrive/3018 project/recommend_index_DL_FINAL.pkl", "rb") as f:
    data = pickle.load(f)

model = data["model"]
embeddings = data["embeddings"]
products = data["products"]

print("Engine loaded! Ready for queries.\n")

def recommend(query, top_n=10):
    q = query.strip()

    # Extract price limit (supports "under 1000", "below $800", "less than 500", etc.)
    price_max = None
    if m := re.search(r"(under|below|less than)\s*\$?(\d+)", q.lower()):
        price_max = float(m.group(2))

    # Extract brand (supports "from Apple", "by Samsung")
    brand = None
    if m := re.search(r"\b(from|by)\s+(\w+)", q.lower()):
        brand = m.group(2).title()

    # Encode query with deep learning
    q_emb = model.encode([q], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, embeddings)[0]

    cand = products.copy()
    cand["dl_score"] = scores

    # PRICE FILTER — only items with real price
    if price_max is not None:
        cand = cand[cand["has_price"] & (cand["price"] <= price_max)]

    # BRAND FILTER
    if brand is not None:
        cand = cand[cand["brand"].str.contains(brand, case=False, na=False)]

    # FINAL SCORE: 92% semantic + 8% popularity
    cand["final_score"] = 0.92 * cand["dl_score"] + 0.08 * cand["popularity"]
    result = cand.sort_values("final_score", ascending=False).head(top_n)

    # Pretty formatting
    result = result.copy()
    result["price_str"] = result.apply(
        lambda r: f"${r.price:.2f}" if r.has_price else "Price not shown", axis=1
    )
    return result[["title", "brand", "price_str", "average_rating", "rating_number", "final_score"]]

# ——————————————————— CHOOSE MODE ———————————————————
mode = "cli"        # Change to "telegram" when you're ready for real users!

if mode == "telegram":
    # TELEGRAM BOT MODE
    !pip install python-telegram-bot --quiet

    from telegram import Update
    from telegram.ext import Application, CommandHandler, MessageHandler, filters

    TOKEN = "YOUR_BOT_TOKEN_HERE"  # ← Paste your token from @BotFather

    async def start(update: Update, context):
        await update.message.reply_text(
            "Amazon Electronics Recommender (Deep Learning)\n\n"
            "Just type what you want!\n\n"
            "Examples:\n"
            "• gaming laptop under 1500\n"
            "• wireless earbuds from Sony under 100\n"
            "• 65w charger for macbook\n"
            "• portable power bank 30000mah"
        )

    async def handle(update: Update, context):
        query = update.message.text
        try:
            recs = recommend(query, top_n=8)
            msg = "Top matches:\n\n"
            for _, r in recs.iterrows():
                msg += f"{r.title.title()}\n"
                msg += f"   {r.brand.title()} • {r.price_str} • {r.average_rating:.1f}★\n\n"
            await update.message.reply_text(msg.strip())
        except Exception as e:
            await update.message.reply_text(f"Oops: {e}")

    def run_bot():
        app = Application.builder().token(TOKEN).build()
        app.add_handler(CommandHandler("start", start))
        app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle))
        print("Telegram bot is LIVE! Go talk to it!")
        app.run_polling()

    threading.Thread(target=run_bot, daemon=True).start()

else:
    # CLI MODE — instant testing (perfect for demo & debugging)
    print("CLI MODE ACTIVE — Type your query below (or 'quit' to exit)\n")
    while True:
        try:
            q = input("Your query: ").strip()
            if q.lower() in ["quit", "q", "exit", "bye"]:
                print("See you later!")
                break
            if not q:
                continue

            print("\nSearching best matches...\n")
            results = recommend(q, top_n=10)

            for i, (_, r) in enumerate(results.iterrows(), 1):
                print(f"{i}. {r.title.title()}")
                print(f"   {r.brand.title()} • {r.price_str} • {r.average_rating:.1f}★ ({r.rating_number:,} reviews)")
                print()

        except KeyboardInterrupt:
            print("\n\nBye!")
            break
        except Exception as e:
            print(f"Error: {e}\n")

Loading Deep Learning recommendation engine...
Engine loaded! Ready for queries.

CLI MODE ACTIVE — Type your query below (or 'quit' to exit)

Your query: gaming laptop under 1500 from Lenovo

Searching best matches...

1. Lenovo Gx30M39704 300 - Mouse - Right And Left-Handed - Wired - Usb - For 320 Touch-15, 320-14, 320-17, 520-22, 520-24, 520-27, 720-18, Legion Y520-15, V110-15 Black
   Lenovo • $7.97 • 4.5★ (26,583 reviews)

2. Lenovo K4 Note Dual Sim, 16Gb Lte Unlocked Phone (Black)
   Lenovo • $-1.00 • 4.0★ (39,970 reviews)

3. Lenovo Chromebook C330 2-In-1 Convertible Laptop, 11.6" Hd Display, Mediatek Mt8173C, 4Gb Ram, 64Gb Storage, Chrome Os, Blizzard White
   Lenovo • $274.00 • 4.5★ (10,961 reviews)

4. Lenovo Tab M10 Plus Tablet, Fhd Android Tablet, Octa-Core Processor, 128Gb Storage, 4Gb Ram, Dual Speakers, Kid Mode, Face Unlock, Android 9 Pie, Iron Grey
   Lenovo • $208.50 • 4.4★ (10,118 reviews)

5. Lenovo Flex 5 14 2-In-1 Laptop, 14.0" Fhd Touch Display, Amd Ryzen 5 4500U